In [17]:
from pathlib import Path

dpath = Path.cwd().resolve().parent / ".cache/price_data.parquet"

In [18]:
import pandas as pd 

data = pd.read_parquet(dpath, engine="pyarrow")
df = data.xs("Close", level="Price", axis=1)


In [19]:
# all empty rows 
df[df.isna().all(axis=1)]

Ticker,TCBI,GREEL,LIF,CRVS,SMCI,ATLO,BVFL,CDNS,AKAM,DYN,...,CEPO,PGNY,BBGI,MXL,AGMB,CTRN,ACHV,QURE,HNRG,WSHP
Date,,,,,,,,,,,,,,,,,,,,,


In [20]:
# any empty rows 
df[df.isna().any(axis=1)].head()

Ticker,TCBI,GREEL,LIF,CRVS,SMCI,ATLO,BVFL,CDNS,AKAM,DYN,...,CEPO,PGNY,BBGI,MXL,AGMB,CTRN,ACHV,QURE,HNRG,WSHP
Date,,,,,,,,,,,,,,,,,,,,,
2024-09-25,69.735428,6.044032,39.049999,5.20,45.814999,16.858919,15.24,271.799988,99.639999,34.880001,...,NaN,16.650000,10.07,14.86,NaN,18.389999,4.38,4.820,8.86,NaN
2024-09-26,70.422623,6.057879,39.470001,5.09,40.240002,16.840424,15.33,274.230011,100.110001,34.619999,...,NaN,16.719999,10.80,15.30,NaN,18.469999,4.44,4.975,8.80,NaN
2024-09-27,70.183601,6.078649,38.810001,5.15,41.973999,17.108612,15.35,274.130005,101.570000,33.830002,...,NaN,16.799999,11.31,15.20,NaN,18.500000,4.77,5.080,9.70,NaN
2024-09-30,71.169571,6.150651,39.349998,5.28,41.639999,16.858919,15.35,271.029999,100.949997,35.919998,...,NaN,16.760000,10.77,14.48,NaN,18.370001,4.74,4.930,9.43,NaN
2024-10-01,70.133797,6.125035,41.009998,5.09,40.549999,16.322538,15.13,263.320007,99.820000,34.759998,...,NaN,16.180000,11.13,12.73,NaN,18.059999,4.80,4.620,9.70,NaN


In [21]:
# WeShop Holdings Limited (ticker WSHP) made its public market debut on the Nasdaq Capital Market 
# on November 14, 2025, via a direct listing. 

# The IPO date for Cantor Equity Partners I, Inc. (ticker: CEPO) was January 7, 2025.

In [22]:
dropped_tickers = df.columns.difference(df.dropna(axis=1).columns)

"WSHP" in dropped_tickers

True

In [23]:
import numpy as np 

ticker_missing_days = df.isna().sum(axis=0)
ticker_missing_days[~(ticker_missing_days.gt(0) & ticker_missing_days.lt(30))] = np.nan 
ticker_missing_days = ticker_missing_days.dropna()
# ticker_missing_days.sort_values()

In [24]:
# WCT IPOd during the look back period, everything else are nan due to price adjustments 

In [25]:
import yfinance as yf 
from datetime import date 

raw = yf.download(list(ticker_missing_days.index), auto_adjust=False, start=date(2024, 9, 1), end=date(2026,9,26))

[*********************100%***********************]  19 of 19 completed


In [26]:
tickers = list(ticker_missing_days.index)
df_patched = df.copy(deep=True)[tickers]
df_patched[df_patched.isna()] = raw.Close[tickers]

In [27]:
df_patched.isna().sum().sort_values().head()

Ticker
CPOP     1
SFWL     1
GECCH    1
GTBP     1
BIOA     1
dtype: int64

In [28]:
# conclusion: replacing autoadjusted nans with raw closes doesnt path the data 

In [ ]:
df_patched = df.ffill()
df_patched.isna().sum().sort_values()[lambda x:x>0].head()

Ticker
BIOA      1
GECCH     1
PRHIZ     4
WCT       5
CBLL     12
dtype: int64

In [ ]:
# BIOA IPO September 27, 2024
# GECCH IPO 
df_patched.CBLL

Date
2024-09-25     NaN
2024-09-26     NaN
2024-09-27     NaN
2024-09-30     NaN
2024-10-01     NaN
              ... 
2026-09-21    7.77
2026-09-22    7.75
2026-09-23    7.52
2026-09-24    7.75
2026-09-25    7.93
Name: WCT, Length: 502, dtype: float64

In [ ]:
# Conclusion: the best way to handle messy data here is to drop any tickers 
# who IPO during your lookback. this can be spotted by simply checking if the 
# first inded of the LB is NA.
# DO NOT drop tickers who have auto aujusted NANs i.e one off days with NAN values. 
# BUT ALSO do not ffill or interpolate this data! it should be handled in operations instead 